In [ ]:
import numpy as np
np.random.seed(0)

# ?Escala tus *pipelines* de procesado de datos mixtos sin salir de scikit-learn!

## Pedro Morales <pedro.morales@ebury.com>
## Enrique Colin <enrique.colin@ebury.com>

![ebury](img/ebury_logo.jpg)
![pycones](img/pycones_logo.png)

### PyConES 2018 @ M?laga, 2018-10-06

# ?Por qu? scikit-learn?

![sklearn](img/sklearn.png)

* Extensa colecci?n de algoritmos y herramientas auxiliares para transformar *datasets* y construir modelos
* API muy expresiva y notaci?n altamente estandarizada
* Permite f?cil integraci?n de funcionalidades personalizadas
* Proyecto muy activo en constante mejora
* **Python**!


# `Pipeline` - ?Para qu? sirve?

![mariopipe](img/mariopipe.jpg)

* En su *camino* hacia el estimador final, someteremos a los datos a transformaciones intermedias como p.ej.:
  - Tratamiento de `NaN`s: `sklearn.impute.SimpleImputer`
  - Encoding a tipos num?ricos: `sklearn.preprocessing.OneHotEncoder`
  - Reducci?n de dimensiones
      + *Feature selection*: `sklearn.feature_selection.SelectFromModel`
      + *Feature extraction*: `sklearn.decomposition.PCA`
  
  
> `sklearn.pipeline.Pipeline` nos ayuda a **encapsular** cadenas de transformaciones **secuenciales** sobre nuestros datos

## Ejemplo: *Titanic dataset*

In [ ]:
import pandas as pd

data = pd.read_csv((
    'https://raw.githubusercontent.com/amueller/'
    'scipy-2017-sklearn/091d371/notebooks/datasets/titanic3.csv')
)

data.head(2)

Supongamos que queremos predecir la probabilidad de supervivencia `survived` con la edad `age` y precio de billete `fare`.

In [ ]:
columns = ['fare', 'age']
target = 'survived'

X = data[columns].values
y = data[target].values

Componemos una cadena de transformaci?n donde:
+ imputamos `NaN`'s, 
+ escalamos las variables, 
+ y finalmente predecimos con una regresi?n log?stica.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

clf = Pipeline(steps=[('imputer', SimpleImputer()),
                      ('scaler', StandardScaler()),
                      ('classifier', LogisticRegression(solver='lbfgs'))])

Esta encapsulaci?n nos permite operar f?cilmente con el modelo completo, trat?ndolo como un ?nico objeto. 

In [ ]:
from sklearn.model_selection import cross_val_score

cv_scores = cross_val_score(clf, X, y, cv=5)
print('CV accuracy: %0.3f' % cv_scores.mean())
print('"0" class prior: %0.3f' % (1 - y.mean()))

## pero adem?s...
La correcta encapsulaci?n de operaciones es **esencial** en la etapa de **validaci?n** (leakage...)

![leakage](img/leakage.jpg)

## `Pipeline` - ?cubre toda la casu?stica de *arquitecturas* de modelos?

## No
* Incorporando **diferentes tipos de datos** al modelo, el preprocesado y extracci?n de *features* consistir? en **varias cadenas** de operaciones **paralelas**, no en una ?nica rama secuencial


![model_example](img/model_example.png)

>## ?Y qu? hay de `sklearn.pipeline.FeatureUnion`?
>
>* No nos da toda la flexibilidad que necesitamos: aplica diferentes cadenas de operaciones (paralelas) al **mismo** conjunto de datos
>* No permite **selectividad** sobre columnas.

# Antes de scikit-learn 0.20

## Alternativa 1: *Do it yourself*

* Desarrollo *propio*, con el apoyo de las clases presentes `sklearn.base` (herencia de `TransformerMixin`,  `BaseEstimator`...) 

![esfuerzo](img/sisifo.jpg)

## Alternativa 2: [sklearn-pandas](https://github.com/scikit-learn-contrib/sklearn-pandas)

>This module provides a bridge between Scikit-Learn's machine learning methods and pandas-style Data Frames.
>
>In particular, it provides:
>
>1. **A way to map DataFrame columns to transformations, which are later recombined into features.**
>
> ...

* Funcionalidad estrella: `DataFrameMapper`
* Constituye la semilla de `sklearn.compose.ColumnTransformer`.

# sklearn 0.20 y la herramienta prometida: `ColumnTransformer`

## Volvamos a nuestro ejemplo...

In [ ]:
data.head(2)

Construyamos un modelo mas complejo, con las siguientes variables:
+ Variables num?ricas:
    - `age`: float.
    - `fare`: float.
+ Variables categ?ricas, discretas:
    - `embarked`: categor?as codificadas como strings {`'C'`, `'S'`, `'Q'`}.
    - `sex`: categor?as codificadas como strings {`'female'`, `'male'`}.
    - `pclass`: enteros ordinales {`1`, `2`, `3`}.

Las transformaciones adecuadas para cada variable depende (entre otras cosas) de su tipo. P.ej.:
- Variables num?ricas:
    1. Imputaci?n de `NaN`s con la mediana
    2. Escalado est?ndar
- Variables categ?ricas:
    1. Imputaci?n de `NaN`s con una nueva categor?a `missing`
    2. *One hot encoding* (dando solo ceros a categor?as no vistas en entrenamiento)

- Variables num?ricas:
    1. Imputaci?n de `NaN`s con la mediana
    2. Escalado est?ndar

In [ ]:
numeric_features = ['age', 'fare']
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())])

- Variables categ?ricas:
    1. Imputaci?n de `NaN`s con una nueva categor?a `missing`
    2. *One hot encoding* (dando solo ceros a categor?as no vistas en entrenamiento)

In [ ]:
from sklearn.preprocessing import OneHotEncoder

categorical_features = ['embarked', 'sex', 'pclass']
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))])

### Encapsulamos las cadenas de preproceso de variables categ?ricas y num?ricas en un unico objeto:

In [ ]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)])

Finalmente, empaquetamos el m?dulo de preproceso con el estimador:

In [ ]:
clf = Pipeline(steps=[('preprocessor', preprocessor),
                      ('classifier', LogisticRegression(solver='lbfgs'))])

Y ya podemos aplicar operaciones selectivas sobre nuestro `DataFrame` de pandas:

In [ ]:
X = data.drop('survived', axis=1)
y = data['survived']

cv_scores = cross_val_score(clf, X, y, cv=5)
print('CV accuracy: %0.3f' % cv_scores.mean())
print('"0" class prior: %0.3f' % (1 - y.mean()))

# ?Preguntas?

# ?Muchas gracias!